# 📋 MNT02 미션 10 — AI 기반 야전 정비 부품 수요예측 및 재고최적화 시스템 통합 패키지 완성

**Solution 교관 전용** / 학생 공개 첨부 금지 / 교육용 가상 자료

---

## 시작 전 꼭 읽으세요

이번 활동은 **교육용 가상 부품 데이터**를 분석하는 실습입니다. 코딩이 처음이어도 안내와 TODO를 따라 한 단계씩 진행하세요.

1. 아래 **준비 셀**을 실행합니다. 필요한 라이브러리는 실습 환경에 설치되어 있어야 합니다.
2. 미션의 입력 자료·핵심 함수를 확인합니다. 학생용 `TODO` 부분을 직접 작성하거나 허용된 AI에 도움을 요청합니다.
3. 코드를 실행하고 실제 출력·오류를 기록합니다. 실행이 실패했다면 실패 내용을 그대로 남기고 수정 후 다시 실행하세요.
4. AI에게 보낸 **실제 질문과 받은 답변 원문**을 작성 기록에 붙여넣고, 코드·출력과 비교한 수정 이유를 작성합니다.
5. 마지막 기록 셀을 실행해서 JSON과 검토용 ZIP을 만들고 내려받습니다. 파일 생성만으로 학습 PASS나 사람의 승인이 되지 않습니다.

**인터넷/폐쇄망:** Colab과 외부 AI는 접속이 허용된 환경에서만 사용합니다.

폐쇄망에서는 사전 설치된 로컬 Jupyter 등 허용된 환경이 필요하며, 외부 AI 사용이 불가능하면 '미실시'로 기록하세요.

**보안:** 실제 부대·조달·업체·정비 자료를 입력하거나 AI로 실제 발주를 승인하지 마세요.

**실습 전 필수:** ① 환경 설정 셀만 실행하면 됩니다.

실습자료(미션 lesson·forms, 공통 안내)가 노트북에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.

준비 완료 문구를 확인한 뒤 다음 셀로 진행하세요.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'MNT02_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
MNT02_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global MNT02_PROJECT_READY, MNT02_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        members = [m for m in archive.infolist() if not m.is_dir()]
        if not members:
            raise ValueError('ZIP 안에 실습자료 파일이 없습니다.')
        if len(members) > 15000 or sum(m.file_size for m in members) > 800*1024*1024:
            raise ValueError('ZIP이 너무 크거나 파일 개수가 너무 많습니다.')
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    MNT02_PROJECT_DIR = WORKSPACE
    MNT02_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 4개: MNT02_Colab_실행안내.md, MNT02_M10_forms.md, MNT02_M10_lesson.md, MNT02_공통_작성안내.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAgIAAAAIVz+D1bE0QYAANcMAAAbAAAATU5UMDJfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kbVdNTxtXFN3zK55UdVkH8lW6RCmqLAFFcbLpBpngSG7B'
    'tmyzSFcGBuRgU0ywwziMydCYzzrN4A88qGbjnzPvzX/oue8920PCAhkzb+4799xzz718x2bnXow/ZM+Sy9FFJgp1//1bJip5vt4eG5sIqacLmezqUiyRDcVT'
    'bxKL/KTHxEHba9lM1HP8xmBiMycOSsKw9HGEEdsdv2zwY0vYNX/bDf0ZT4kaTpoNfnrLC3leqIeYsA3RbAvbEkd7vOWy38LzXqvHvE6DXxt+5WJwznMdcdxm'
    'Im8Ku8L83TbfKfPtMhNHDYXYr7T5rsH4zr/ifVvsO+IshyS2AUKHGHsYYn7FEB/KfsVivNYTtTZ3i/wj4DT4WZEJoy5uzMCdidWV1Jt+NxVNLEUz/W7mVZy+'
    '4uOPePaH5Vg0neh3V6LZ1HIyuxxflLmtN8SHi0AIUXX5dY5IRYoA5+BS33D63an5MPPX6/2uODqR/B04XpOeDjGa32bwKMT4F1dsmmwmlskkcf3rZHolI2om'
    'EeabBvNahqjcSii1WyoPOBU1l04QNttgfrHEC+cAEEa9GD879lwXh3vyHbsSxK5AT4UZ8Ijm536XFzq8lQPm5j9MtPPQSL/rdXvelcN4MSeOkRdvtUXVwJGO'
    'Jf7eBKe2v1Gj2PTguD0KP/Z4kAxvGVoGbCn2mqQVX0ktx1agtmg2nkww3nDFUQl6I8TQiTCuUG2WjqWWo69iC69XE6/onLB6EBtC5cBpMA/U+ZQg1SgMELsO'
    'WPGcd6IGgi0bPN8jtNsy37dIYaHFGFiOUWiVAy+ZUvhHJr/8j5cswsebFVxgWPxyk6KCEZSdl4ZR55LZ8CCp2NJ0Op1MM7Gx5m9YqmuklKWOSQKivocajV4f'
    'e0LS7YhCAx/n6IFBuf1Dg6VXEwsr8UyGGKAaNk1RK7JhUzDICh1FtH5dAmKGIgxSvi57Tg6ncl7zFkrqofqSUEkF0gNPe/jEU8gsGPfG8DqlQFzUVQMkGMVz'
    'aRfGFd8+kXfVc0S+rtY9naq7XLapUsh1Dk0T7PUhM09DLLmaTa1mMw9meTPvm+4D/0NROOeklUx2CQ/RstmlWDpNyAmySqLr8tNevwv2Qr9niDoQAKUC5ahq'
    'fHcTVsX45Tnfz8u6OjlUDUBK/FORsvKcNXzRb6Jn+ZeczKpypKHLzpME6qzkRXh/t6oZkNHtwXFSuL0GE4HKQIzUxR1uvFbH3+roKsmCHzTgYQOQw/cJzEV7'
    'UE3eLAsLPcCbBrVaTRadH9cCTP4Ip/+RibM1Ye8B4Sa3TwKuPsjcrqDa6C8UHDd/9is2cnjkXbVJTHTCa+b8LTsoMviOZUM+Zl6swyFQIr9qgi6mrINo9Kvg'
    'xQVkaqyZ6annc9PPF8Jz8y9f4OISImi49xkJo8AFi0XCsy9npl5M/0zKhFSk6FQ2yvOG/mCJa+s+fx04nnJxWVdYxmkNtDmW+ETh9mSH4r2DrYAvT4K6SVT6'
    '0fj491Qxf/ctENEfuOMQlXeGnbTZagXTSlQhKdVNhKrqjh5JcekhKFmBPnihQXer3yiVr2vIqFEqEJx7LJVGzdyhjPx3eX55IecEBgRVAanQr7DGC2eY9+EN'
    'KlEm2XjOlSjYw0DAXDF4oTpI+CckPDFO8Uh2O672eSo++JIXq9ZRYwB8QD9kLnfCYK60hbkmtQUg+9bEuOcAxtGJ7Bo9M1su/BWAMSDJdbqW17yQcwaskHNS'
    'mfVMGEni2oJ80UJUDcrcOVfLAp3DUDbQvt/CoTlDUu80MMpkLutSReg3sg29YKy3IQ5qMtyHaIplgx4p6Y/qPTYxLueJnMG5iXE9uwcqyJskpoA6Dku0AWkq'
    'Zx/3u7NP8TOpJjPJQ/d0ZHpm+hmEvjAbjkTCv85FSEcUXEbUDScFpPUhBQSnF1s72rugbBNdJPEbw/Yb8vdsNiL3kLz5QMPRpaIdaaAsmbN6NZgzNkZ+lifv'
    'OnsrDQSH9fBSQ4EGJcWUo1NTopZOhZfEKdtW8UnmAr+WNgDRw7gpXb9YxNLCKAfV6ncsZBU7WXQJI4F20cHiiR2P738OzL5BidXs1bsOFhn0QEBgENMFHiNZ'
    'eFelSlutlDObn4pEAKxT8tpQ0vaN3LMM7Rn3+4uepMPYtIbJujfkq8F1We+4cknTjge63tVJgNyuc+cwYEBjSjtgbiPPfolnmf+Xq7X5cDL0BGbkOUZAaqiv'
    'bLGtIi0T6rFenOQ8uZaDh+9foRxyyfhGIQPLQAQshyr+SBXKN4KCkv9b8I+Ov4FLDNoeUWwQLbtBDi21tanVSgXX/5CgxzFytrErS1lqgxo5TdmRtn5m3N0M'
    '/gdQSwMEFAAAAAgAAAAhXFigt/EuCAAANBMAABIAAABNTlQwMl9NMTBfZm9ybXMubWTdWF1vG8cVfdevGMDom2R+xGgBIUXhPBQxYDsF7LeiEFybTgXUkkAp'
    'AgIIASUtDUZkIMrlSqSyFMhYsiiHhlfkSloBch78U5y3ndn/0HPvnV2SNuK8tEURWJDInZ2ZO+eec+4dX1N37t7P5tWdXFaZgx3jnCqzu2OqbfWu1FA3b6ko'
    '9LXfVMa9Mh1HmY6rLx2lz0vxs4oylabZb5hmxYRdpf1trNCPhh1z5pnOetxqKFP1zNZhXPZU/PQsdk9UXOvFG4fmuKRMy8FmU1Pm+7oehire/jbyS7Pqr7Hr'
    'YNm/fTiBA8IGsdsyW2cYqJl2EDcc3b4y7ZAiM5vr2PEw3v02Gl6p6KxuBgHeq+tqbxoz+avp1KMBdsdZYveVfnmi/nLz3j0M7wemua4QA1bV//IoPpzUeD2V'
    'y2Z/N610WNMvsPkOn/qyiTftZioa+NjQtPAAsVw2Y7eptxp44frUlB6WosEbReMdb1bdvv/Z3J0v7t7//F7+xrunz3J5JS+YNkC9dBAzJgNCZZxOvNnWdU/p'
    'lz39Xfj2QtdKyIXZC95emON109lJ8pXLmoMjpfuhcTy89l6c+uhKYarp+rHr6eohYrp2zaILJMyL9tTUmuLhV1iZTwTAoqGj1uiACNvGYuOQ02Iw2TG6uIpO'
    'feM52A2PCc/Mn2/eup3Rr0NaoOrRy6Po0wiRMkrxGvafmZlRv/gb42ri5//3AWEroon3vwM0vqd9goiBZi11gfeMMht9piCTMxUe1NKpz6p3u11aRFfPQA4Q'
    'JMA8ZiwN2KdAUh/UVPy9Y2dj0Zu3zF5d6eNuFEIPe2XdD4SUgfE6s7Rrx4uGSHyH1KCf15DvNw2QHX+RZfcN5RV6FP5aWtM8CBQSwzqKtL65DiaNkx6KPV5n'
    'nh4FNBGHA5lmiVdIO/kAdAZHSAmlyDUGr2gvkEjpZp2ndUpE5JRI9t3kwB99N4mwqfRJA5F8jFRTH8kh/IhDA0t5W9MupTFzACzUYWC6AairqxVSlBIUrZST'
    'EJOcM06sQr8JiyHsaIGWY1HQ244lxmhFgg4MMM6hfllOzyy7iBoJnySHrNm3F3G5Rma7FSpR5AQswjROMNOB5nsd+Crmwyc6Lk48DPQRAw///qFs4xWjvcJr'
    'Fuv/hlqnPolOA45sc50RtoY1QpjPqvSgEjdDAtP4PwnIm8T56CJEbFQl9HGd7DN2A129VHojMPsn5OgbzSSPcDaz4SewjeUHDEVhGcuA5JPJjvxsHQHiWg3l'
    'Br4WOwIvJ5wLoxWSHjj0NR1jgb4/ht/8fDyCMd4L/DbPSbr+A7CTOdlyS158WOJt7W4wgHOPTi2VOKmOa8mr/jMc4NyBtmBnoR/5o/pgT5BUAjp8tQ9LwoTX'
    'IT4SbnVQDG1DhZZeSwTwv3F+Mj4cMqikibBfSWFBxbgVq2ibAnbKJGXlEILS5+ssD+ptpOKF+gXOq7sePlrsxmaDQKYNtqL+lZvGu8J7SjtNsmVqMEhsI3NF'
    'zY+CEtlF/BTGDHfd9KQGSM2Uj+OOwU/M1iWk/XOpoatEdQgE50L7UU0r8s8lV2q9e2aqfezTj5tOWoOQaLcH6bORlHUH7MYh8FfFLZcaKjwf/KgScxGTUGO0'
    'HXmv70WDPsnENA/hIOnLCR0+asUP/7G4uFyYWy0Ul+cXFzDA/9A4vQ5txLRfrccO6JxChiNGckVyHV1tpT5BITunqHJjnaA5p2ZsQpE8Nt50jZyXEGt5erul'
    '0QKZY4eP/14BX7OvqLUELYlpTBJ89pFr2sQyjX6F+hjJ4TN15PAq9BGzttFW0mlPj7pgfHQrcBPpu3m3eA9Od1zCDPSs1R4bOdy0vTNNiTLPGQ4z8LhFHviY'
    'OerNCBCuT4MKuaqgZLuB9/txdkzu4W1fSTOoed9vMBUsp8QqqJz2EBwiROOzf0IyoExKx0OhTNIAFQsLWD6SYbvcIUs9tB1yWtukl5IC9/YCTRAgP3CSnAmE'
    'NNsW2d2KNOd/skxbU/kEbebbrFqaXyr8c36hkHmysJLNzxULK8UH8wvXl76mQjoZKFCQVkSCklAhSrlOSMtMsjB+r5CbBvWoyE48zSvcA9RqLrOa544a15S9'
    '4DoVZ8CIn6xURfu6Pq5N3hMK+U//WMhRhTsJ1Gqes8KFw2YFb1cydx/ctbZO4+Al3d/sPcM2N1JsJDSyK7pbkVCYcXYtGApwNOcnpntqWQG9idwSnxBLhOPv'
    'eBmxSUyxDdMIeVn9dmF5WXLNAk1vQvbWwRYMVtm+6cO8fZLkjRP9W7s72tSkArK1F30ReiXxWNzuup7U1iSdXNkyYsCpy1nGjBAEaoB95Ub+mxt/QIubMFJI'
    'ABlgR6aUr7fL+sWbpEkQDnH0k67LTQVSheqILVZztrckryV3IigF+ZRgCbGYRegL5PS89EHF7AYTvGR1TEzadiBlRfV4eDhq2jgFSGQ+m//9TDaXkpvdExtR'
    'QeYrELnT2BTJWi7L+oMrse9Q62+v1sOQrh5yz0prTgXwsIEd7MhNGBvQtXds7eQGw8N2/uj2zgUcNjVe/ynIoYPjja1CbcrzKwSSETSoV0ZSpcKQ3nhmUlmw'
    '/kHfOveHkrlhJUMMfV6bVY8WHy5bm1tcKhQfrMDTlue+/Gr+UeH6k0fTv2yF0+NT5xdWCsXHDx4W5paXCg95YrGwtFhcSV74e3G+8Hh+4UseerxYfLKc4f92'
    'mkMsc/wdI2QEovwEKGaddJ58jxQfkZKKYUZD+mPK4p4PMel6Mzlz0nxS1sOuXHEkUXiQbCEbciqkDLKju6jq8UEFJoYEfwDjvwFQSwMEFAAAAAgAAAAhXDXY'
    'gYb2CQAAMxsAABMAAABNTlQwMl9NMTBfbGVzc29uLm1k7VnfTxvZFX73X3GlqFIrOYCdtFuhVqu03baRknTV5KlVBSxxdlGzgIAi9aUyZIgccIpJPcGQMTt0'
    'za+U1U7MgI0KL/wpm7e5d/6HfuecO+PxsqHPWa2EBIzv3Dn3O9/5znfG19Tdew+GiupuYUh9U66rW7dV1A100FDGPTO+o4zv6lNH6ZNy/KKiTKVhNuumUTHd'
    'baWDVWW2DqMj3xx7xl+IN+rKrHhmuRUveSp+ehy7Byqu7seLLbNXVmbDMc6bXO7aNWWeYPFBdHSm9OuDeM3L5aKgjIvJY/TzwDTD2AlwATc1lV4J9cqp2arp'
    'bc84no3joqNX9s2/l3i/J95FR6JRUecsehPonTN7O+LQKxW90hqwNyo5QZ7Cj90Ns3yMP90Kzis7YCfPidcBwl4Zd+SVfY6ulk1zDf8GnvnyTJmTuml7dIyo'
    'HeBO+hzomfXQNB0Vu57S7YpZrwGVVvzyWew29HL9EjAUp2CDOyhIgehVTR91KV7ciX3qiMhslFW8+gxY5XLXkxWx62CR0l93zVIDB8QZ2iE9EafCs4EWPVT2'
    '6QGR3o+jPqmo303MqfifXcSgbgz8FEmMAudtuU67dLej4AWyAYAcvdJ4W3Z7z5ZdhxUl5ssq0tDEDtjRd/Ek2sz4tahdBriboWkgva/39fMuAZYCb68pkCw6'
    'rqn4laPwm44QV2uAnbBvmWYVvw5t2vj+bH6UWQzMiZeAHjdretcx3hnIkkLPyNo8YhuiCZgG+pkmqAn+Ir3R8aHea/34zoNf/YSOpr94hi3T5PeWEw3aIRP1'
    'KDSer8z6IZjLiUS57OJoDXoA4YHIWmvZR+EIiwHWecY9t0TiG5lLsRte5hIRqHGodwNhsEBJBdMM9b88VRiKApxza4eJKHVk1oPoCFV7UKeCRPFFHS9qHxDh'
    'GVvJCyoPp+tWOeI1RZXtnhOKAFP7Owkdr6s/C8n+crmoWTR6yYyrVSrcuqObZ6bZJTg4HE4Cx9ef275wiLtI4FdASn186/79/OWzok5AFOPt49RDP+oLHoV9'
    '2sBK+zCCEQ+kkjGI5bQh1YcFAxn+Mm/04dmwejg1Pjv4+eTcUHFkaro0MzY3MTU5O/Lp3yYelgY+f8haNz0xXXo8MVmyy2ZKczNjE5MD03+/DKKUsA9tOLbF'
    'QRWL/wAgLqxxbbgVvRhedH499XjskyTq5KLUr9JL3Xi5q08WkisELSTQb+Ky+tPtj/kw53VgM6xG3xng6ABwPTaooNjdJ2bvUCYO44ZDxVoGdxLGtv+TREKV'
    'eOs243dCsIHr5ah93ichqU4Pq77ARh+OzY0N/vGjW7+5+xHQG2Up3kc/oWSc4xh2ixsfgFic18VDJP+iU/h5cuFVDcy12F10igXiuJUKs7UEdhJPqWBPnH5V'
    'A9+3kIaXa2alCUy4vY2gvY08mpr5fJai4WRyvmdK01Mzc7OmKfrJIpykJ14BUySP4F67YhWW1losBUFKLNP+7MoM0CrGHqvGP5uami2NzJdmZsGyUWaGA9YM'
    'W8In52y0TLBfKuSRCghN/9Wi+ubpCzVfGJwvKn3YpUJaD6EOno+yxc8QoubS4+V6r2oV1ia6VPzFL0sFOtlBqOaLrECOHz9ppkp5Whm8N3bP9hsmDFoq2rMl'
    'Ctjoe/YemxiWW9ABxBGh7gkIDIVGKfT6urSKhI7UoGyxMJcpw7Zh3y0Q4uOflcb/2ltmg5gfezwBppV6dO6jQl+amM9Mb+cNP4hZTiQiDd8O7ZYkudthP+UV'
    '6YjvMUa2jZJkObRjAmrPYAgw8FEQcDgieJSjLkmhmBJFsvuUDEajpXcaLFu+S7Ahc1wmi/vR6UJyCKJfIq0bnl7doCvX7N8FFl9QJ958zj2Ri8hqay5nNU4c'
    '1kWHbB1JHgNMVokqi8/UPOeMZls08QjyPihyzSIg55Jm3OvF3C1ehvrlf6N2ldtOO4hdPzqtfkh7SywZeKAoaGd6bzvqdq2rSWIhKaKmeNGJN1xCTAcHuu3y'
    'E9hQZEBJICha08oR8slsznK5dwjeZaGTRsHYWyZgkxctuytgA7xWbHA/d2N/wWweZE7Vx5NhBQMxcvcP9x78/n7xJqq0ULQLmG3ZKpTy0bVE2kCWpOsDjb0F'
    '468lclQYgkpzpTvI3rf7HkcPV7IdZDxkAtIN4YnwT1YxXluHggCoYlnICIqNyTaFZGGW5tnC7wvbBnrURazctbitsaHIxE2E72/LH1LNGKelXy/hNsmlPOVG'
    '9CZUImdMhoQtqalITs7RUhI3gPPKMUDXqw5TFq1ESPXdNLopNkZcoHBWCsrag7SYRA1ZJVplKm3Ou2hBErW06m9rQEhjkmPB9bBtlj5fd1FRWIUOg5Ly3XQL'
    's3zKNornG0vivo5HgxsEcu5m8R83P4C/TMpX0oPVMPKs8oFeXdK75+kh+SBsl8jWb6R55XOmc8N8wSoByacdLfp8O/V8q/Us4/qkInnlrbcq0Ia+VsENq++m'
    'VfgGRAp0jlo9Z5AY2eJQ8WfXhwppv+HxDQ8iqLgVZFpF1hJzS9yxvCPfnaWlral0qKkAHmscpMDwgN/eun0ns3cif/yxvb9XopwlSOxxLQrLVuIpyCMHx+vt'
    'YuU8xEiKxe2y2W0q2H/kDwY4l/shle9ZKqWQWSkwl2AAFl0gRzh69TxBnu9qs5jdYGJyrjTzaGy8NDI7XRpPbrfm1a75ZGai9Ghi8lP7KZtdezgglc/4EppQ'
    '2l/hxM01iGSeaRRW8H+2CeST7ogERx0CpNPVu2fUozO+JztKYfD94plFAGhl5i8IFhjGDgQUxm8k13fzPBwidf6ClVzbomhTgZ+cn41bEpERQkkIWU4yNtvZ'
    'dzwYFVg0OScsuJxU+0IAZgj8AF35nZS73COqdftcVWzaGrY90HGsJ0kyXGDlteZr+L14rUR30By/WactZIjqJYVCYS8GE7l5QAWol3e4kjCbI5T+mYUka2tJ'
    'StIOL9yauX2njZh8L61kXyo5vujAVnnAw0kUUCBkgyWJfFmxhiBXtCCz8Rp+9/wtE0I2Phw+a2EkQnqh9X2cr96WySSbkwOz/caSAfZW5hyhCNJryxjsH5RS'
    'xi3W76aA293vlGZnJcU8BaXeVWKyE9p6DTMdsMyk64ZNF6d1+Hv20shmJC0Xq2nwnXCnIp6w4dseguRpRbLI0jooyppOnJYoPdh/aPzvT+PnDgrPtrUz2JtO'
    'kFT7nQPKzA74MofwdxJWp3uVclMqRTrU/3v3mL/qxeNVLiGv3mkR8mIQBi+/HaOylzpP8GGyRcELnvIwzYhqkIifNvAxgyDvLyh5/AZA19Kh7jtmG5sfXEge'
    'IQ/kDEivY/120XPjrYpt2D30cplvLq784mJQtpTvL9KvLwbsSx6Z1nFeeVOgF0M0PrnInOT3I8t1qZ3M3Pg/UEsDBBQAAAgIAAAAIVyxBwko6AYAAJ8NAAAc'
    'AAAATU5UMDJf6rO17Ya1X+yekeyEseyViOuCtC5tZG1WXU8bRxR9968YKe+YkI+mjyihLVJAUT4UqS8RTWmCRHEERFX7tIY1MrZTTPDGa7LrLI2NgS7qYhts'
    'FPPin+OZ/Q89d2bWXj4kPizvzp1zzz333HuLzcw+H59gg9ZpuHHKRHVbmCdMWFm+1k4kBoEh1tNMZG2+7/MzQ2x8YPxDINx2aAZ8z6EnYrck7Kzo7vU7ouoP'
    'Wh7jZ6Yw3X6H5w/EPxmGCOG6wwZNI9zw8FLuXLhdNugigCtck4l8TeROQ8vmuVIsugwS2hYLSyauEp4b5rqimuH5LM/Xxpj4XOStrnBtFn7M8qPDfmf22Wy/'
    '88Ozh7gDsM5tiQApUNzBaVE4e2L3kKmc4oFwv+cAtcELRr8zaJnCukCMckY025RUnZ8DZbXIvxZ4vcfoqFcHXtEwwFQO4HWoROLWLTY5zcKKg5skNMsEFrDb'
    'E56VSNweY4vzKyupJVzTDUQN5y9KfMeRYd0LSR7lhJ82olj4FFpgec0HSEBxbe53+x1cIRxTVE3GmyV8QrSmIfZdxoND3rSIVH6AGxHeS4/gMZ4/5S0j9pi3'
    'a8Lp8UNctu2IvBNah1EqE2MsgpivhZ82Ub8AaRBS3mqLvTYIILyT06Jc1Oz0O6ENWL7OCoA9R3H5uRjVt2UMmhey7N7lG6M6kJ6axzjT/DcGGGhRiBi+O2NS'
    'fJ4VozAsFITbIzw8sIWdDq02kPkqgdhNxN/XHuLhFnmYCdML1112Z3CCE26bBOKaQB3lRaJYC5BfpiByNRJihdQi9oIYpLuAVLX50TdeVHKChKEBoqti6rwk'
    '+cMSq3bTTOp0YkUCIXlnqIMRUY4H6SmVexZU3mrzug1E4tShhlPJS4mc27JgaweD83QE817UOyDV4l82dZ0RTxVAFRph/+vyvC//o7vlTdQ78ka86RYYPcnY'
    'TJQDnCHEqqljUF++/TP5U+qP5Mu3c6sov8qcXgt3P4wuDBweOMmhQagswkKRDAQEQtmDk2DUseRLaV12SXDkGbAaF3wKF8reyvD9C61dKSDHpFYaNNfxLj8M'
    'NHgqc9YRpoNqkstoqbThh6AZ/VtB/ygHo+Qhh62K8EwQw4sk6MGpDxHgg7DwZWXYaVLmI2aVynjTxFnqVtlKuZIOXy2SS0lONapyUTTSwttW1MZrrysn9Tpz'
    't9+ZuY/fB3yLIsOvyCrARbpNIZAVgGXtS+qXJqUQKIdqw2cTCRUYUNez7MeFVRb+3dVUTjwYuwcdDgLzmvsycn569JF4RALdPfoMS9lzOBhc82+ySdihJxs3'
    '74AVpI4+yIpPbSTB0b4oT7fAv/gYOVIiZD5hxSbbKBd5KcA7uIQeSUx4HcXAuUaWGgKRYoZ3dAhjY0rJyGXQ6ao6XxMqoqPHMSZ8xJTeVOkCI3qPUIFFyS4b'
    'fVXvUf/nvZvnAMiFZJPU+p6VVNVKhlublCzuxh0kgDOTHx1QpKPMFSuEWMsUlyYv2kYNUrq57pE9kdayEDEGEwYN+rFgkNQkISzcPVDq0tNbNW2sdlppeV9b'
    'rh7icDTdNLttMs8ozLD/b5yZGL4stLsoEORCmipnuA+iGtuy8Qw13fWg4sGneBtDiGobwDnJ+ZkTCXHmO9n5st9IrbBxQHk2PfPi8eTzqUdaeHFEul+OQ8vT'
    'IxplQj2vtBnF8mhERmPceTw1+XR26umr6dknL55H2jXp8FWRUB2yGBbqEoxihBTlNglrtF7gxjNTeUw00dCTWAeqGJOw/X03quDILaCCHaA+w4RwKJrKVxXV'
    'hRd5k9OkQcUGzQPI8KbGgo1isA0tlLA0alIeEC9VCjKmSrmkCBJPtQ7kugg0vXUd9bIm+/f4ci6yrkPLN0R5I657vcvFNrafp5+A7IepxblfyJJsn3u18KOB'
    'nHimi+f8LK284/KIxsTE4A0/mxxt3TB4Y1NkDOVp798tpuZ+XUnKxXXsr4V3OM8bBb5zPDwMxJerrkYlih1tJlQ805HUhJVS5DPDHQcVkSSRJLnfi+lsYWll'
    'dfn969XUMhNBl5xMjxgDr9IBSAJrGtL/5g/HDxE4hINaji6lE5J9SiJmKNf3SorSRAVlaaPNV5o0dY+aNIlEnPqhhcccXMobmz72pDjbUKRHywq1BqiS7+rj'
    'tMJlEUStduqeawrWmWwUyEjkaeIxD2s+J0J0hobqq3jXeg4mLeaNxO3jr9qtNLVq586VVlLvl1/Pv0otL7xZWJpbXEm+WVh9tTz/2/zy/BK+nxifuH97fPx7'
    '0scl/yey1rH4HDydmnw0M4XAKouF39+llldp6O6coMV1Ulenm1retXtTgeXeRYts0b46bV3j9jjtMfXe1XnLoi+caOOw7+BNNQzQXphRVHtYwHbx6qqX+B9Q'
    'SwECFAMUAAAICAAAACFc/g9WxNEGAADXDAAAGwAAAAAAAAAAAAAApAEAAAAATU5UMDJfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kUEsBAhQDFAAAAAgAAAAhXFig'
    't/EuCAAANBMAABIAAAAAAAAAAAAAAKQBCgcAAE1OVDAyX00xMF9mb3Jtcy5tZFBLAQIUAxQAAAAIAAAAIVw12IGG9gkAADMbAAATAAAAAAAAAAAAAACkAWgP'
    'AABNTlQwMl9NMTBfbGVzc29uLm1kUEsBAhQDFAAACAgAAAAhXLEHCSjoBgAAnw0AABwAAAAAAAAAAAAAAKQBjxkAAE1OVDAyX+qzte2GtV/snpHshLHslYjr'
    'grQubWRQSwUGAAAAAAQABAAUAQAAsSAAAAAA'
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 상황: 어떤 문제를 해결하나요?

**미션 개요**

수요 예측, 재학습, 안전재고·위험 지수, 단종 대응, 발주 추천 및 결재 대기열 생성에 이르는 모든 시스템 기능을 한 번에 실행하는 통합 패키지를 완성하는 실습입니다.

**배경 및 필요성**

개별 미션에서 구축한 데이터 전처리, 수요 예측, 재고 최적화, 단종 대응 및 HITL 승인 워크플로우를 유기적으로 결합하여 실제 현장에서 즉시 구동 가능한 완전한 자동화·통합 시스템의 형태로 묶어낼 필요가 있습니다.

자동 재학습 파이프라인의 성능 비교 및 교체 판단, 자체 점검, 그리고 완비된 운영 가이드를 통해 시스템의 신뢰성과 지속 가능성을 확보해야 합니다.

**핵심 과업**

- 통합 파이프라인 실행: 자료 생성 및 정합성 점검, 운영 모델과 재학습 모델의 성능 비교 후 교체 판단, 다음 달 예측 및 위험 지수 산출을 단일 흐름으로 실행합니다.
- 단종 대응 및 예산 제약 발주 추천: 단종예고 품목의 최종 구매(LTB) 수량 산출, 단종 품목의 대체품 보유 여부 정리, 생산중 품목의 예산 제약 발주 추천을 수행하여 정비장교 결재 대기열에 올립니다.
- 종합 산출물 및 가이드 완성: 운영 가이드 10개 장, 가상 연계 메시지 규격, 자체 점검표, 그리고 브리핑 요약을 최종 출력합니다.

**검증 및 산출물**

**검증 기준:** 재학습 파이프라인 정상 실행 및 교체 판단 완료, 자체 점검 전 항목 PASS, 운영 가이드 완비율 100%, 브리핑 수치와 실제 실행 결과의 일치 여부 확인.

**주요 산출물:**

docs/mnt02_operations_guide.md, pipeline/mnt02_retrain.py, 브리핑 요약.

## 📁 이번 미션에서 완성할 것

- **미션에서 완성할 내용**
  - 자료 생성·정합성 점검, 운영 모델과 재학습 모델 비교 후 교체 판단, 다음 달 예측과 위험 지수 산출을 한 흐름으로 실행한다.
  - 단종예고 품목은 최종 구매(LTB) 수량을, 단종 품목은 대체품 보유 여부를 정리하고 생산중 품목은 예산 제약 발주를 추천해 결재 대기열에 올린다.
  - 운영 가이드 10개 장과 가상 연계 메시지 규격, 자체 점검표, 브리핑 요약을 출력한다.
- **구현 계약**
  - 핵심 구현 계약 pipeline/mnt02_retrain.py / choose_version: 운영모델 오차e1,후보모델 오차e2 → v1/v2 문자열.
  - 유한한0이상 오차만 비교하고 e2<=e1이면 v2를 선택한다.
  - 빈/NaN 평가를 거부하고 실제 선택 모델로 이후 예측한다.
- **변경 과제와 검증**
  - 추가 변경·검증 수요 예측, 재학습, 안전재고·위험 지수, 단종 대응, 발주 추천과 결재 대기열을 한 번에 실행하는 통합 패키지를 완성한다.
  - LTB_MONTHS24→12 변경을 비교하고 선택된 모델·대기열·직접 작성 10장 문서·브리핑 수치를 대조한다.
  - 핵심구현 후첫성공실행을보존하고추가변경·재실행·실제AI원문·근거검토3건·보완·최종판단을 작성합니다.

**제출 경로:** `docs/mnt02_operations_guide.md`, `pipeline/mnt02_retrain.py`, `docs/mnt02_interface_spec.md`, `reports/mnt02_briefing.md`, `forms/MNT02_M10_forms.md`


## 🔍 교관 검토 기준

### 정량 평가와 측정 근거

- [정량적 평가] [필수] 통합 패키지 — 재학습 파이프라인 정상 실행과 교체 판단, 자체 점검 전 항목 PASS, 운영 가이드 완비율 100%, 브리핑 수치가 실행 결과와 일치하는가.
- PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다.
- 목표미달·0기준·미측정은그대로표시합니다.
- 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

### 정성 평가와 검증 근거 · 원자료 대조와 검토

- [정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.
- PASS 증거 M10의t42～47은모델 선택 구간이며독립 최종 평가가 아닙니다.
- 원본에서는v1 유지 후전체 학습 모델로예측하는 불일치가 있어실제 선택v1/v2로예측하도록 고쳤습니다.
- 가상 2026-01 실제 수요는미제공입니다.

### 정성 평가와 검증 근거 · 판정 상태와 적용 범위

- 가이드 10문장 출력은 10장 본문 작성이 아니며 시작문서는FAIL입니다.
- 실제 문서 작성·직접 승인·교관 검토는별도입니다.
- 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

### 진행 추적 및 완료 기준

- [진행 추적 및 완료 기준] 정성·검증: M10의t42～47은모델 선택 구간이며독립 최종 평가가 아닙니다.
- 원본에서는v1 유지 후전체 학습 모델로예측하는 불일치가 있어실제 선택v1/v2로예측하도록 고쳤습니다.
- 가상 2026-01 실제 수요는미제공입니다.
- 가이드 10문장 출력은 10장 본문 작성이 아니며 시작문서는FAIL입니다.
- 실제 문서 작성·직접 승인·교관 검토는별도입니다.

### 제출 파일 및 완료 기록

- 파일: docs/mnt02_operations_guide.md, pipeline/mnt02_retrain.py, docs/mnt02_interface_spec.md, reports/mnt02_briefing.md, forms/MNT02_M10_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
- 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
- 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

> **주의:** 아래 구현 코드는 원본 교관용 Workbook의 참고 구현입니다. 실행 결과와 기준 달성 여부는 실제 환경에서 독립 검증해야 합니다. 승인·발주·실측 사실을 생성한 것이 아닙니다.


## 📋 순서대로 할 활동

### 활동 1: 계획과 사전 판단

- 원문 목표 및 운영 모델 유지·교체 전략 기획 원본 목표 및 기본 제공 코드를 철저히 분석하여, 수요 예측, 재학습, 안전재고·위험 지수, 단종 대응, 발주 추천과 결재 대기열을 유기적으로 연결하는 통합 패키지의 전체 아키텍처를 기획합니다.
- 기존 운영 모델을 유지할지 새로운 후보 모델로 교체할지 판단하는 기준을 수립하고, 실제 다음달 예측 작업과 어떻게 연동할지 구체적인 프로세스를 설계합니다.
- AI 위임 범위 및 검증 계획 설정 파이프라인 통합 및 자동 재학습 과정에서 AI에 위임할 코드 작성 범위와 활용할 자료를 명확히 확정합니다.
- 입력 시점, 기간, 단위, 그리고 사용 가능 시점의 정합성을 검증할 계획을 세우고, choose_version 함수 등의 입출력 계약(Contract) 조건을 철저히 정의하여 사전 준비를 마칩니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M10 계획: 수요 예측, 재학습, 안전재고·위험 지수, 단종 대응, 발주 추천과 결재 대기열을 한 번에 실행하는 통합 패키지를 완성한다.에 필요한 입력·기간·단위·사용 가능 시점과 choose_version의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 2: AI와 코드 변경

- **함수 구현과 첫 실행**
  - 핵심 함수(choose_version) 구현 및 첫 실행 보존 운영 모델 오차 e1과 후보 모델 오차 e2를 입력받아 문자열 v1 또는 v2를 반환하는 핵심 함수 choose_version을 구현하고 저장한 뒤, 첫 성공 실행 결과를 온전히 보존합니다.
  - 유한한 0 이상 오차만 비교하여 e2 <= e1일 때 v2를 선택하고, 빈 값(NaN) 평가는 엄격히 거부하여 실제 선택된 모델을 기준으로 이후 예측을 수행하도록 설계합니다.
- **AI 활용과 변경 과제**
  - AI 기획 반영 및 변경 과제 수행 앞서 수립한 파이프라인 통합 기획안을 바탕으로 구현된 코드를 검증하고 AI와 함께 고도화 작업을 진행합니다.
  - 변경 과제: LTB_MONTHS 24 → 12 변경 사항을 상호 비교하고, 선택된 모델, 대기열, 직접 작성한 10장 규모의 문서 및 브리핑 수치를 서로 대조합니다.
  - 실제 코드와 출력 결과물은 모두 안전하게 저장합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M10 구현: pipeline/mnt02_retrain.py의 choose_version를저장하고계약〔운영모델 오차e1,후보모델 오차e2 → v1/v2 문자열.
- 유한한0이상 오차만 비교하고 e2<=e1이면 v2를 선택한다.
- 빈/NaN 평가를 거부하고 실제 선택 모델로 이후 예측한다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 3: 근거 대조와 재실행

- **변경 코드 재실행**
  - 수정 코드 저장 및 다각도 성능 경계 시험 통합 패키지 및 재학습 파이프라인 관련 수정 코드를 안전하게 저장하고 스크립트를 재실행합니다.
  - 선택 모델, 대기열, 10장 본문 문서, 실행 출력, 그리고 브리핑 수치가 서로 정확히 일치하는지 다각도로 상호 비교하여 검증합니다.
- **AI 설명과 원자료 대조**
  - AI 설명 대조 분석 및 주의사항 준수 AI가 제시한 설명 및 코드 선택 사항 3건 이상을 실제 실습 자료, 실행 결과 로그와 상호 대조하여 타당성을 검토합니다.
  - t42 ～ t47 구간은 모델 선택 구간이며 독립 최종 평가가 아니라는 점을 명확히 인지합니다.
  - 원본의 v1 유지 후 전체 학습 모델 예측의 불일치를 실제 선택 v1/v2로 올바르게 고쳤는지 재확인합니다.
  - 가상 2026-01 실제 수요는 미제공 상태이며, 가이드 10문장 출력은 10장 본문 작성이 아니므로 시작 문서는 FAIL 처리됨을 유념합니다.
  - 실제 문서 작성, 직접 승인, 교관 검토는 별도로 수행되어야 합니다.

#### 확인할 사항:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M10 검증: [필수] 통합 패키지 — 재학습 파이프라인 정상 실행과 교체 판단, 자체 점검 전 항목 PASS, 운영 가이드 완비율 100%, 브리핑 수치가 실행 결과와 일치하는가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - M10의t42～47은모델 선택 구간이며독립 최종 평가가 아닙니다.
  - 원본에서는v1 유지 후전체 학습 모델로예측하는 불일치가 있어실제 선택v1/v2로예측하도록 고쳤습니다.
  - 가상 2026-01 실제 수요는미제공입니다.
  - 가이드 10문장 출력은 10장 본문 작성이 아니며 시작문서는FAIL입니다.
  - 실제 문서 작성·직접 승인·교관 검토는별도입니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 4: 최종 판단과 산출물

- **산출물 제출과 상태 기록**
  - 종합 산출물 제출 및 상태 명시 원본 평가 기준에 따른 최종 결과물인 docs/mnt02_operations_guide.md, pipeline/mnt02_retrain.py, docs/mnt02_interface_spec.md, reports/mnt02_briefing.md, forms/MNT02_M10_forms.md의 실제 작성 내용, 최종 측정값, AI 대화 보존 위치, 선택 이유를 상호 연결하여 체계적으로 제출합니다.
  - 미달 항목, 미측정 항목, 문서 미작성 내용, 그리고 실제 승인 대기 상태를 명확히 구분하여 기록합니다.
- **미션별 검증 원칙**
  - 모델 선택 구간 특성 및 검증 원칙 명시 t42 ～ t47 구간은 모델 선택 구간이며 독립 최종 평가가 아니라는 점을 명확히 인지합니다.
  - 원본의 v1 유지 후 전체 학습 모델 예측의 불일치를 실제 선택 v1/v2로 올바르게 고쳤는지 재확인합니다.
  - 가상 2026-01 실제 수요는 미제공 상태이며, 가이드 10문장 출력은 10장 본문 작성이 아니므로 시작 문서는 FAIL 처리됨을 유념합니다.
  - 실제 문서 작성, 직접 승인, 교관 검토는 별도로 수행되어야 함을 최종 검토합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M10 완료: docs/mnt02_operations_guide.md, pipeline/mnt02_retrain.py, docs/mnt02_interface_spec.md, reports/mnt02_briefing.md, forms/MNT02_M10_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

## 이 미션의 AI 활용 예시

- 가상 자료 분석·기획·코드 변경·디버깅·검증 설계·해석
- "교육용 미션 10의 핵심 함수 구현에서 누락되기 쉬운 입력값·경계조건을 알려주세요. 원본 자료에 없는 성능 수치를 만들지 말아 주세요."

AI 답변은 그대로 정답이 아닙니다. **실제 실행 로그와 최소 3건의 주장·선택을 비교**하세요.


## 원본 교관용 코드 — 참고 구현

다음은 제공된 Solution의 해당 미션 코드입니다. 실제 수행 시 환경에 필요한 패키지를 확인하고 실행하세요.


In [ ]:
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """수리부속 청구 이력 생성·정합성 점검, 수요 예측 모델 자동 재학습(기존·신규 모델 비교 후 교체), 안전재고·결품(단종) 위험 지수, 단종예고 품목 최종 구매량과 단종 품목 대체 검토, 예산 제약 발주 추천과 정비장교 결재 대기열을 한 번에 실행하는 통합 패키지를 완성하고, 자체 점검표·운영 가이드 목차·체계 연계 메시지 예시·브리핑 요약을 출력한다."""
    import json
    import time
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.stats import norm
    from scipy.optimize import linprog
    from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
    from sklearn.metrics import mean_absolute_error

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 야전 정비창 수리부속 청구/출고·장비 가동률 데이터 생성기
    # 실제 부대·품목·재고·단가 정보와 무관하다. 품목 40개 x 48개월(가상 2022-01 ~ 2025-12) 월별 자료를 만든다.
    # ---------------------------------------------------------------------------
    FSC = {"2520": "power_train", "2530": "brake_steer", "2910": "fuel", "2930": "cooling",
           "2940": "filter", "5330": "gasket", "6140": "battery", "5961": "semiconductor"}
    EXERCISE = {3, 8, 10}      # 가상 대규모 훈련 월(훈련 계획으로 미리 알 수 있음)
    N_MONTHS = 48


    def season(fsc, month):
        """품목군별 계절 계수: 축전지는 겨울, 냉각계통은 여름에 수요가 늘어난다."""
        if fsc == "6140":
            return 1.6 if month in (12, 1, 2) else 0.9
        if fsc == "2930":
            return 1.5 if month in (6, 7, 8) else 0.9
        return 1 + 0.1 * np.sin(2 * np.pi * month / 12)


    def make_items(n=40, seed=2):
        """품목 마스터: NSN(가상), 품목군, 장비 유형, 기준 수요, 단가, 조달 리드타임, 공급업체 수, 생산(단종) 상태."""
        rng = np.random.default_rng(seed)
        rows = []
        for i in range(n):
            f = str(rng.choice(list(FSC)))
            status = str(rng.choice(["생산중", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
            rows.append({"item_id": f"P{i + 1:03d}", "nsn": f"{f}-00-{rng.integers(100, 1000)}-{rng.integers(1000, 10000)}",
                         "fsc": f, "group": FSC[f], "equip_type": str(rng.choice(["A", "B", "C"])),
                         "base": float(np.round(rng.lognormal(2.7, 0.8), 1)),
                         "price": int(rng.lognormal(12, 1.0)) // 100 * 100,
                         "lead_time": int(rng.integers(1, 7) if status == "생산중" else rng.integers(4, 13)),
                         "suppliers": int(rng.integers(1, 5)) if status == "생산중" else 1,
                         "status": status, "alt_available": bool(rng.random() < (0.6 if status == "생산중" else 0.3))})
        return pd.DataFrame(rows)


    def make_history(items, months=N_MONTHS, seed=3):
        """월별 청구 수요(개), 긴급 불출 건수, 장비 유형별 운용시간·가동률."""
        rng = np.random.default_rng(seed)
        t = np.arange(months)
        moy = t % 12 + 1
        ex = np.isin(moy, list(EXERCISE))
        hours = {e: (100 + 20 * rng.random(months)) * np.where(ex, 1.5, 1.0) for e in "ABC"}
        avail = {e: np.clip(0.93 - 0.12 * (hours[e] - 100) / 80 + rng.normal(0, 0.015, months), 0.6, 0.99) for e in "ABC"}
        rows = []
        for it in items.itertuples():
            trend = 1 + rng.uniform(-0.1, 0.3) * t / months
            s = np.array([season(it.fsc, m) for m in moy])
            lam = it.base * s * trend * hours[it.equip_type] / 110
            d = rng.poisson(lam)
            rows.append(pd.DataFrame({"item_id": it.item_id, "t": t, "ym": [f"{2022 + k // 12}-{k % 12 + 1:02d}" for k in t],
                                      "month": moy, "exercise": ex.astype(int), "op_hours": hours[it.equip_type].round(1),
                                      "availability": avail[it.equip_type].round(3), "demand": d,
                                      "emergency": rng.binomial(d, 0.05)}))
        return pd.concat(rows, ignore_index=True)


    def make_stock(items, hist, seed=4):
        """현재(가상 2026-01) 재고: 보유량과 미입고 발주량(조달 중인 물량). 단종 품목은 추가 발주가 어려워 미입고량이 적다."""
        rng = np.random.default_rng(seed)
        recent = hist[hist["t"] >= hist["t"].max() - 5].groupby("item_id")["demand"].mean()
        st = items[["item_id", "lead_time", "status"]].copy()
        st["on_hand"] = [int(round(recent[i] * rng.uniform(0.5, 3.0))) for i in st["item_id"]]
        cover = np.where(st["status"] == "생산중", rng.uniform(0.6, 1.3, len(st)), rng.uniform(0.0, 0.6, len(st)))
        st["on_order"] = [int(round(recent[i] * L * c)) for i, L, c in zip(st["item_id"], st["lead_time"], cover)]
        return st[["item_id", "on_hand", "on_order"]]
    # ---------------------------------------------------------------------------

    # ---- 수요 예측 공통 함수(파생 피처 + 머신러닝 모델) ----
    FEATS = ["lag1", "lag2", "lag3", "lag12", "ma3", "ma6", "std6", "season_idx", "month_sin", "month_cos",
             "exercise", "op_hours", "op_ratio", "item_level", "trend_t"]


    def make_features(hist):
        """품목별 과거 수요로 파생 피처를 만든다(모두 예측 시점 이전 정보만 사용)."""
        df = hist.sort_values(["item_id", "t"]).reset_index(drop=True)
        g = df.groupby("item_id")["demand"]
        for k in (1, 2, 3, 12):
            df[f"lag{k}"] = g.shift(k)
        df["ma3"] = g.transform(lambda s: s.shift(1).rolling(3).mean())
        df["ma6"] = g.transform(lambda s: s.shift(1).rolling(6).mean())
        df["std6"] = g.transform(lambda s: s.shift(1).rolling(6).std())
        df["item_level"] = g.transform(lambda s: s.shift(1).rolling(12).mean())
        df["season_idx"] = df["lag12"] / (g.transform(lambda s: s.shift(12).rolling(12).mean()) + 1)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
        df["op_ratio"] = df["op_hours"] / df.groupby("item_id")["op_hours"].transform(lambda s: s.shift(1).rolling(12).mean())
        df["trend_t"] = df["t"]
        return df.dropna().reset_index(drop=True)


    def new_model():
        """Prophet/LSTM 대신 사용하는 포아송 손실 그래디언트 부스팅(개수 자료에 적합)."""
        return HistGradientBoostingRegressor(loss="poisson", max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                             min_samples_leaf=20, random_state=0)


    def mape(y, p, min_y=1):
        y, p = np.asarray(y, float), np.asarray(p, float)
        m = y >= min_y
        return float(np.mean(np.abs(y[m] - p[m]) / y[m]))

    # ---- 앙상블 수요 예측·안전재고·결품(단종) 위험 지수 공통 함수 ----
    STATUS_W = {"생산중": 0.0, "단종예고": 0.5, "단종": 1.0}     # [교육용가상자료] 단종 상태 가중치


    def fit_ens(tr):
        """포아송 그래디언트 부스팅(70%) + 랜덤포레스트(30%) 앙상블."""
        h = new_model().fit(tr[FEATS], tr["demand"])
        r = RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=0, n_jobs=1).fit(tr[FEATS], tr["demand"])
        return h, r


    def pred_ens(m, X, w=0.7):
        return np.clip(w * m[0].predict(X[FEATS]) + (1 - w) * m[1].predict(X[FEATS]), 0, None)


    def next_month(hist, items, t_new=48, op_mult=1.0, ex=0):
        """다음 달(t_new) 예측용 행: 작년 같은 달 운용시간 x 배수, 훈련 여부 지정."""
        last = hist[hist["t"] == t_new - 1].copy()
        prev = hist[hist["t"] == t_new - 12].set_index("item_id")["op_hours"]
        m = (t_new % 12) + 1
        f = last.assign(t=t_new, ym=f"{2022 + t_new // 12}-{m:02d}", month=m, demand=0, exercise=ex,
                        op_hours=last["item_id"].map(prev).to_numpy() * op_mult)
        X = make_features(pd.concat([hist, f], ignore_index=True))
        return X[X["t"] == t_new].set_index("item_id").loc[items["item_id"]].reset_index()


    def stock_policy(items, mu, sigma, z=1.65, review=1):
        """안전재고 = z x 월 예측오차 표준편차 x sqrt(리드타임+검토주기), 재주문점 = 월 수요 x (리드타임+검토주기) + 안전재고."""
        L = items["lead_time"].to_numpy() + review
        ss = z * sigma * np.sqrt(L)
        return pd.DataFrame({"item_id": items["item_id"], "mu": mu, "sigma": sigma, "L": L,
                             "ss": np.ceil(ss), "rop": np.ceil(mu * L + ss)})


    def risk_index(items, pol, stock):
        """결품 위험 지수(0~100): 리드타임 동안 수요가 가용 재고를 넘을 확률(정규 근사)을 기본으로,
        단종 상태·단일 공급업체·대체품 부재에 가중을 준다(가상 가중치)."""
        st = stock.set_index("item_id").loc[items["item_id"]]
        avail = (st["on_hand"] + st["on_order"]).to_numpy()
        p = 1 - norm.cdf((avail - pol["mu"].to_numpy() * pol["L"].to_numpy()) / (pol["sigma"].to_numpy() * np.sqrt(pol["L"].to_numpy()) + 1e-6))
        sw = items["status"].map(STATUS_W).to_numpy()
        sup = (items["suppliers"].to_numpy() == 1).astype(float)
        alt = (~items["alt_available"].to_numpy()).astype(float)
        score = 100 * np.clip(0.55 * p + 0.25 * sw + 0.10 * sup + 0.10 * alt, 0, 1)
        return pd.DataFrame({"item_id": items["item_id"], "p_short": p, "status_w": sw, "single_sup": sup, "no_alt": alt,
                             "avail": avail, "risk": score.round(1)})

    LTB_MONTHS = 24        # [교육용가상자료] 단종예고 품목 최종 구매(LTB) 대상 기간(개월)
    t_all = time.perf_counter()
    checks = []

    # 1) 자료 수집·정합성
    items = make_items()
    hist = make_history(items)
    stock = make_stock(items, hist)
    ok = (hist["demand"] >= 0).all() and items["nsn"].str.match(r"^\d{4}-\d{2}-\d{3}-\d{4}$").all() and not hist.duplicated(["item_id", "t"]).any()
    checks.append(("원천 자료 정합성(NSN 형식·음수·중복)", ok))
    df = make_features(hist)
    print(f"[1] 품목 {len(items)}개 x {N_MONTHS}개월 청구 이력, 학습 행 {len(df)}개, 정합성 {'통과' if ok else '실패'}")

    # 2) 재학습 파이프라인: 운영 모델 v1(t<36) vs 재학습 v2(t<42), 최근 6개월(t=42~47) 검증 후 교체 판단
    va = df[df["t"] >= 42]
    big = va["item_id"].isin(items.loc[items["base"] >= 10, "item_id"]).to_numpy()
    v1, v2 = fit_ens(df[df["t"] < 36]), fit_ens(df[df["t"] < 42])
    p1, p2 = pred_ens(v1, va), pred_ens(v2, va)
    e1, e2 = mape(va["demand"][big], p1[big]), mape(va["demand"][big], p2[big])
    eb = mape(va["demand"][big], va["ma3"][big])
    use = "v2" if e2 <= e1 else "v1"
    print(f"[2] 검증 MAPE(수요 많은 품목): MA3 {eb:.1%}, 운영 v1 {e1:.1%}, 재학습 v2 {e2:.1%} -> {use} {'교체' if use == 'v2' else '유지'}")
    checks += [("자동 재학습 실행·비교", True), ("운영 모델 MAPE 25% 이하", min(e1, e2) <= 0.25)]

    # 3) 다음 달 예측(전체 자료로 최종 학습) -> 안전재고·위험 지수
    final = fit_ens(df)
    mu = pred_ens(final, next_month(hist, items))
    sg = np.maximum(0.25 * mu, np.sqrt(np.maximum(mu, 1)))
    pol = stock_policy(items, mu, sg)
    rk = risk_index(items, pol, stock)
    alert = rk["risk"] >= 60
    print(f"[3] 다음 달 예측 수요 {mu.sum():.0f}개, 결품 위험 경보(60 이상) {int(alert.sum())}개 품목, 안전재고 합 {pol['ss'].sum():.0f}개")

    # 4) 단종 대응: 단종예고 -> 최종 구매(LTB) 수량, 단종 -> 대체품·재생정비 검토
    ip = (stock["on_hand"] + stock["on_order"]).to_numpy()
    eol = items.assign(mu=mu, ip=ip, risk=rk["risk"].to_numpy())
    ltb = eol[eol["status"] == "단종예고"].assign(ltb=lambda d: np.maximum(0, np.ceil(d["mu"] * LTB_MONTHS - d["ip"])))
    dis = eol[eol["status"] == "단종"]
    print(f"[4] 단종예고 {len(ltb)}개 품목 최종 구매 추천 합 {ltb['ltb'].sum():.0f}개(향후 {LTB_MONTHS}개월분), 단종 {len(dis)}개 품목 중 대체품 보유 {int(dis['alt_available'].sum())}개")
    for _, r in ltb.sort_values("risk", ascending=False).head(3).iterrows():
        print(f"    {r['item_id']}({r['group']}) 월 {r['mu']:.1f}개, 가용 {r['ip']:.0f}개 -> 최종 구매 {r['ltb']:.0f}개")

    # 5) 예산 제약 발주 추천(LP, 생산중 품목. 단종예고는 최종 구매로 별도 처리) -> 결재 대기열
    gap = np.where(items["status"] == "생산중", np.maximum(0, np.ceil(pol["rop"].to_numpy() - ip)), 0)
    price = items["price"].to_numpy().astype(float)
    idx = np.where(gap > 0)[0]
    budget = 0.6 * float((gap * price).sum())
    lp = linprog(c=-(rk["risk"].to_numpy()[idx] + 10) / gap[idx], A_ub=[price[idx]], b_ub=[budget], bounds=[(0, g) for g in gap[idx]], method="highs")
    q = np.zeros(len(items)); q[idx] = np.floor(lp.x)
    queue = [{"id": f"Q{k + 1:02d}", "item": items.loc[i, "item_id"], "qty": int(q[i]), "risk": float(rk.loc[i, "risk"]), "status": "결재대기"} for k, i in enumerate(np.where(q > 0)[0])]
    queue += [{"id": f"Q{len(queue) + k + 1:02d}", "item": r["item_id"], "qty": int(r["ltb"]), "risk": float(r["risk"]), "status": "결재대기", "type": "LTB"} for k, (_, r) in enumerate(ltb[ltb["ltb"] > 0].iterrows())]
    auto = [x for x in queue if x["status"] != "결재대기"]
    print(f"[5] 발주 추천 {int((q > 0).sum())}건(예산 {budget / 1e6:.1f}백만 중 {np.dot(q, price) / 1e6:.1f}백만) + 최종 구매 {int((ltb['ltb'] > 0).sum())}건 -> 정비장교 결재 대기 {len(queue)}건, 자동 확정 {len(auto)}건")
    checks.append(("자동 확정 0건(HITL)", not auto))

    # 6) 운영 가이드·연계 메시지
    GUIDE = {"1. 목적·범위": "야전 정비 부품 수요 예측·재고 최적화·단종 대응", "2. 데이터 규격": f"품목 마스터 {items.shape[1]}개 필드, NSN 가상 형식",
             "3. 전처리": "NSN 정규화, 결측 월 계절성 보간, 긴급 불출 보존", "4. 예측 모델": f"포아송 GBM+RF 앙상블, MAPE {min(e1, e2):.1%}",
             "5. 안전재고·위험 지수": "z=1.65, 리드타임 결품 확률+단종·공급업체·대체품", "6. 단종 대응": f"LTB {LTB_MONTHS}개월분, 대체품 호환성 검증",
             "7. 결재선·감사": "자동 확정 금지, 승인·조정·반려·보류 사유 기록", "8. 재학습": "분기별 재학습, 검증 MAPE가 낮을 때만 교체",
             "9. 폭증 대응": "작전 템포 계수 보정, 경계 구간 '주의' 표시", "10. 연계 인터페이스": "ORDER_REQ 가상 메시지 규격"}
    done = np.mean([bool(v) for v in GUIDE.values()])
    checks.append(("운영 가이드 10개 장 완비", done == 1))
    print("[6] 운영 가이드 목차:", " / ".join(GUIDE))
    top = max(queue, key=lambda x: x["risk"]) if queue else {}
    msg = {"msg_type": "ORDER_REQ", "version": "1.0", "item_id": top.get("item"), "nsn": items.set_index("item_id").loc[top.get("item"), "nsn"] if top else None,
           "qty": top.get("qty"), "risk_index": top.get("risk"), "model_ver": use, "approval": "PENDING", "source": "교육용가상자료"}
    print("[7] 체계 연계 메시지 예시(가상 규격):", json.dumps(msg, ensure_ascii=False))
    elapsed = time.perf_counter() - t_all
    checks.append(("통합 실행 60초 이내", elapsed < 60))

    print("\n[자체 점검표]")
    for name, okk in checks:
        print(f"  [{'PASS' if okk else 'FAIL'}] {name}")
    print(f"점검 {sum(o for _, o in checks)}/{len(checks)} 통과, 운영 가이드 완비율 {done:.0%}, 전체 실행 {elapsed:.1f}s")
    print("\n[브리핑 요약]")
    print(f"- 부품 수요 예측 MAPE {min(e1, e2):.1%}(이동평균 {eb:.1%})이며, 재학습 모델은 검증 후 교체한다({use} 사용).")
    print(f"- 결품 위험 경보 {int(alert.sum())}개 품목 중 단종·단종예고가 {int((alert & (items['status'] != '생산중')).sum())}개로, 단종 대응이 재고 위험의 핵심이다.")
    print(f"- 단종예고 {len(ltb)}개 품목의 최종 구매 {ltb['ltb'].sum():.0f}개와 발주 {int((q > 0).sum())}건을 결재 대기열에 올렸다.")
    print("- 모든 발주·최종 구매·대체품 전환은 정비장교 결재 후 집행하며, 수치는 교육용 가상 자료 기준이다.")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].bar(["MA3", "model v1", "model v2"], [eb, e1, e2], color=["gray", "tab:blue", "tab:green"]); ax[0].axhline(0.25, c="r", ls="--")
    ax[0].set_title("Validation MAPE (t=42-47)")
    col = items["status"].map({"생산중": "tab:green", "단종예고": "tab:orange", "단종": "tab:red"})
    ax[1].scatter(items["lead_time"], rk["risk"], c=col, s=20 + mu / 2); ax[1].axhline(60, c="r", ls="--")
    ax[1].set_xlabel("lead time (months)"); ax[1].set_ylabel("risk index"); ax[1].set_title("Risk vs lead time (size=demand, red=discontinued)")
    plt.tight_layout(); plt.show()


## 원본 교관용 예상 출력 (참고용)

아래 내용은 원본에 포함된 예시이며 **실제 실행 결과로 제출할 수 없습니다.**

<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[1] 품목 40개 x 48개월 청구 이력, 학습 행 1000개, 정합성 통과
[2] 검증 MAPE(수요 많은 품목): MA3 32.7%, 운영 v1 20.0%, 재학습 v2 20.9% -> v1 유지
[3] 다음 달 예측 수요 948개, 결품 위험 경보(60 이상) 13개 품목, 안전재고 합 1108개
[4] 단종예고 8개 품목 최종 구매 추천 합 2779개(향후 24개월분), 단종 3개 품목 중 대체품 보유 2개
    P004(filter) 월 8.1개, 가용 12개 -> 최종 구매 183개
    P019(filter) 월 11.5개, 가용 59개 -> 최종 구매 219개
    P025(filter) 월 14.0개, 가용 89개 -> 최종 구매 247개
[5] 발주 추천 10건(예산 31.2백만 중 30.9백만) + 최종 구매 8건 -> 정비장교 결재 대기 18건, 자동 확정 0건
[6] 운영 가이드 목차: 1. 목적·범위 / 2. 데이터 규격 / 3. 전처리 / 4. 예측 모델 / 5. 안전재고·위험 지수 / 6. 단종 대응 / 7. 결재선·감사 / 8. 재학습 / 9. 폭증 대응 / 10. 연계 인터페이스
[7] 체계 연계 메시지 예시(가상 규격): {"msg_type": "ORDER_REQ", "version": "1.0", "item_id": "P004", "nsn": "2940-00-559-8893", "qty": 183, "risk_index": 87.5, "model_ver": "v1", "approval": "PENDING", "source": "교육용가상자료"}

[자체 점검표]
  [PASS] 원천 자료 정합성(NSN 형식·음수·중복)
  [PASS] 자동 재학습 실행·비교
  [PASS] 운영 모델 MAPE 25% 이하
  [PASS] 자동 확정 0건(HITL)
  [PASS] 운영 가이드 10개 장 완비
  [PASS] 통합 실행 60초 이내
점검 6/6 통과, 운영 가이드 완비율 100%, 전체 실행 3.5s

[브리핑 요약]
- 부품 수요 예측 MAPE 20.0%(이동평균 32.7%)이며, 재학습 모델은 검증 후 교체한다(v1 사용).
- 결품 위험 경보 13개 품목 중 단종·단종예고가 11개로, 단종 대응이 재고 위험의 핵심이다.
- 단종예고 8개 품목의 최종 구매 2779개와 발주 10건을 결재 대기열에 올렸다.
- 모든 발주·최종 구매·대체품 전환은 정비장교 결재 후 집행하며, 수치는 교육용 가상 자료 기준이다.
```
</details>


## 교관 검토 기록

학생의 실제 실행 로그·대화 원문·수정 이력·평가 기준 충족 여부를 확인하고, 달성/미달/추가확인 및 승인 여부를 별도 기록합니다.

**통합 입력·저장 기능:** 마지막 통합 화면에서 미션별 산출물을 파일별로 작성·수정·저장하고, 실제 AI 질문·답변 원문, 원자료·출력 대조 근거, 검토 3건, 수정과 최종 수행 상태를 기록한 뒤 ZIP으로 다운로드합니다.

저장만으로 PASS나 승인이 되지 않습니다.


In [ ]:
# @title ⑤ 교관용 개별 산출물·검토기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 10
        DELIVERABLE_PATHS = ['docs/mnt02_interface_spec.md', 'docs/mnt02_operations_guide.md', 'forms/MNT02_M10_forms.md', 'pipeline/mnt02_retrain.py', 'reports/mnt02_briefing.md']
        PROJECT_NAME = 'MNT02'
        NOTEBOOK_ROLE = '교관용 Solution'
        WORK_ROOT = Path(MNT02_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'MNT02_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'MNT02_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                state['dirty'] = change['new'] != state['saved_value']
                state['saved'] = state['loaded'] and not state['dirty']
                if state['dirty']:
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
                elif state['saved']:
                    state['status'].value = '<span style="color:#286">저장된 내용과 같습니다.</span>'
                else:
                    state['status'].value = '<span style="color:#777">아직 저장하지 않았습니다.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','실습 계획·검토 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE, 'notebook_role': NOTEBOOK_ROLE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# MNT02 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('실습 계획·검토 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if record_dirty['value'] or any(v['dirty'] for v in editors.values()):
                    print('미저장 변경이 있습니다. 먼저 저장 후 ZIP을 다시 만드세요.')
                    return
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ MNT02 미션 {MISSION_CODE:02d} ({NOTEBOOK_ROLE}) 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 교관용 해설 및 예상 출력은 실제 실행·승인 증거가 아닙니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
